Importing Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

Uploading files

In [2]:
leases=pd.read_csv('Data in CSV/leases.csv')
properties=pd.read_csv('Data in CSV/properties.csv',delimiter=';')
locations=pd.read_csv('Data in CSV/locations.csv',delimiter=';')
units=pd.read_csv('Data in CSV/units.csv',delimiter=';')
tenants=pd.read_csv('Data in CSV/tenants.csv',delimiter=';')

In [3]:
tables=[leases,properties,locations,units,tenants]

rename   rent_per_month  

leases.rename(columns={'  rent_per_month  ':'rent_per_month'},inplace=True)

Checking missing values

In [4]:
for table in tables:
    data=table.isna().sum()
    missing=data[data>0]
    for col,count in missing.items():
        print(f" {col} : {count}")

 end_date : 5
 email : 1


Check for duplicates

In [5]:
for table in tables:
    duplicates=table.duplicated().sum()
    print(duplicates)

0
0
0
0
0


Changing rent and arrears to absolute values

In [6]:
leases['rent_per_month']=leases['rent_per_month'].abs()
leases['arrears']=leases['arrears'].abs()

Updating missing  end dates to todays date and Converting to appropriate date types

In [10]:
today=pd.Timestamp.today().normalize()
leases['end_date']=pd.to_datetime(leases['end_date'].fillna(today))

In [11]:
strip=lambda x:x.strip()
leases['start_date']=leases['start_date'].apply(strip)
leases['start_date']=pd.to_datetime(leases['start_date'],format="%d/%m/%Y")

In [12]:
leases.dtypes

id                         int64
unit_id                    int64
tenant_id                  int64
rent_per_month             int64
arrears                    int64
start_date        datetime64[ns]
end_date          datetime64[ns]
dtype: object

In [13]:
leases=leases.astype({
    'arrears':float,
    'rent_per_month':float
}).copy()

Calculated fields : lease duration in months,annual_rent,lease_status,valid_lease

In [14]:
months=(leases['end_date'].dt.year-leases['start_date'].dt.year)*12 + (leases['end_date'].dt.month-leases['start_date'].dt.month)
partials=leases['end_date'].dt.day>leases['start_date'].dt.day
leases['total_months']=months+partials.astype(int)
leases

,id,unit_id,tenant_id,rent_per_month,arrears,start_date,end_date,total_months
0,1,1,1,45000.0,0.0,2024-01-01,2026-05-05,29
1,2,2,2,55000.0,5000.0,2023-11-01,2024-10-31,12
2,3,3,3,65000.0,2000.0,2024-02-01,2026-05-05,28
3,4,4,4,30000.0,1000.0,2024-03-01,2026-05-05,27
4,5,5,3,70000.0,0.0,2025-02-01,2025-12-31,11
5,6,6,5,40000.0,8000.0,2024-06-15,2025-06-14,12
6,7,7,6,52000.0,0.0,2024-08-01,2024-07-31,0
7,8,8,7,38000.0,12000.0,2023-09-01,2026-05-05,33
8,9,9,8,60000.0,0.0,2025-01-01,2026-05-05,17
9,10,10,2,45000.0,1000.0,2024-02-01,2024-12-31,11


Annual rent

In [15]:
leases['annual_rent']=leases['rent_per_month']*12

Active / Inactive leases

In [16]:
mapper=lambda x:1 if ((x['start_date'] < x['end_date']) | (pd.isna(x['end_date']))) & (x['start_date']<=today) else 0

In [17]:
leases['valid_lease']=leases.apply(mapper,axis=1)

In [18]:
leases['rent_paid']=leases['rent_per_month']*leases['total_months']

In [20]:
conditions = [
    (leases['start_date'] < leases['end_date']) & 
    (leases['start_date'] <= today) & 
    (leases['end_date'] >= today),

    (leases['start_date'] < leases['end_date']) &
    (leases['end_date'] <= today),

    (leases['start_date'] > leases['end_date']) | (leases['start_date'] > today)
]
choices=['ongoing','expired','invalid']
leases['lease_status']=np.select(conditions,choices,default='invalid')

In [21]:
leases

,id,unit_id,tenant_id,rent_per_month,arrears,start_date,end_date,total_months,annual_rent,valid_lease,rent_paid,lease_status
0,1,1,1,45000.0,0.0,2024-01-01,2026-05-05,29,540000.0,1,1305000.0,ongoing
1,2,2,2,55000.0,5000.0,2023-11-01,2024-10-31,12,660000.0,1,660000.0,expired
2,3,3,3,65000.0,2000.0,2024-02-01,2026-05-05,28,780000.0,1,1820000.0,ongoing
3,4,4,4,30000.0,1000.0,2024-03-01,2026-05-05,27,360000.0,1,810000.0,ongoing
4,5,5,3,70000.0,0.0,2025-02-01,2025-12-31,11,840000.0,1,770000.0,expired
5,6,6,5,40000.0,8000.0,2024-06-15,2025-06-14,12,480000.0,1,480000.0,expired
6,7,7,6,52000.0,0.0,2024-08-01,2024-07-31,0,624000.0,0,0.0,invalid
7,8,8,7,38000.0,12000.0,2023-09-01,2026-05-05,33,456000.0,1,1254000.0,ongoing
8,9,9,8,60000.0,0.0,2025-01-01,2026-05-05,17,720000.0,1,1020000.0,ongoing
9,10,10,2,45000.0,1000.0,2024-02-01,2024-12-31,11,540000.0,1,495000.0,expired


Convert to Proper Case

In [22]:
locations['name']=locations['name'].str.strip().str.lower().str.title()

In [23]:
locations

,id,name
0,1,Nairobi Cbd
1,2,Westlands
2,3,Kilimani


Saving the cleaned data to a csv file

In [25]:
leases.to_csv("Data in CSV/CleanedLeases.csv")

PERFOMING JOINS WITH RELATED TABLES TO ALLOW WHOLESOME ANALYSIS

 lease with units

In [29]:
lease_units=pd.merge(leases,units,left_on='unit_id',right_on='id',suffixes=['','_units']).drop('id_units',axis=1) 

 lease with units with properties

In [30]:
lease_units_property=pd.merge(lease_units,properties,left_on='property_id',right_on='id',suffixes=['','_property']).drop('property_id',axis=1)


 lease with units with properties with location

In [31]:
lease_location=pd.merge(lease_units_property,locations,left_on='location_id',right_on='id').drop(['id_x','id_y','location_id','id_property'],axis=1)


In [32]:
lease_location.rename(columns={'name_y':'location','name_x':'unit_name','name_property':'property_name'},inplace=True)

A:Total Rent Billed vs Total Arrears by Location. 

In [52]:
rent_arrears=lease_location[lease_location['valid_lease']==1].groupby('location')[['rent_per_month','arrears']].sum()
rent_arrears['paid']=rent_arrears['rent_per_month']-rent_arrears['arrears']
rent_arrears.reset_index(inplace=True)
rent_arrears

,location,rent_per_month,arrears,paid
0,Kilimani,78000.0,3000.0,75000.0
1,Nairobi Cbd,265000.0,8000.0,257000.0
2,Westlands,183000.0,21000.0,162000.0


B:Occupancy Rate per Property. 

perform right join on units to get all units and properties even those without leases

In [69]:
lease_units_all=pd.merge(leases,units,left_on='unit_id',right_on='id',how='right',suffixes=['_leases','_units']).drop('id_units',axis=1) 


In [68]:
lease_units_property_all=pd.merge(lease_units_all,properties,left_on='property_id',right_on='id',suffixes=['_unit','_property']).drop(['property_id','id','location_id'],axis=1)


In [73]:
lease_units_property_all.rename(columns={'name_unit':'unit_name','name_property':'property_name'},inplace=True)


In [84]:
property_unit_leases=lease_units_property_all.groupby('property_name')[['unit_name','id_leases']].count()
property_unit_leases.columns=['Total Units','Occupied Units']
property_unit_leases['perc%']=np.divide(property_unit_leases['Occupied Units'],property_unit_leases['Total Units'])*100
property_unit_leases.reset_index(inplace=True)
property_unit_leases

,property_name,Total Units,Occupied Units,perc%
0,Delta Corner,5,5,100.0
1,Kimathi House,3,0,0.0
2,NSSF Towers,5,5,100.0
3,Riverside Court,3,0,0.0
4,The Junction Residences,4,2,50.0


C:op 3 Properties by Arrears. 

In [85]:
property_arrears=lease_units_property.groupby('name_property')['arrears'].sum().reset_index()
property_arrears

,name_property,arrears
0,Delta Corner,21000.0
1,NSSF Towers,8000.0
2,The Junction Residences,3000.0


D:Average Monthly Rent per Property and per Location.

In [86]:
lease_units_property_location=pd.merge(lease_units_property,locations,left_on='location_id',right_on='id')


In [87]:
lease_units_property_location.rename(columns={'name_y':'location'},inplace=True)

In [88]:
avg_rent=lease_units_property_location.groupby(['name_property','location'])['rent_per_month'].mean().reset_index().rename(columns={'rent_per_month':'Average_monthly_rent'})

In [89]:
avg_rent.sort_values(by='Average_monthly_rent',ascending=False)

,name_property,location,Average_monthly_rent
1,NSSF Towers,Nairobi Cbd,53000.0
0,Delta Corner,Westlands,47000.0
2,The Junction Residences,Kilimani,39000.0


PERSONAL POSSIBLE ANALYSIS

1:SIZE to RENT comparison in different locations

In [91]:
size_rent=lease_units_property_location.groupby('location')[['size','rent_per_month']].mean().reset_index()
size_rent.sort_values(by='size',ascending=False,inplace=True)
size_rent['price_per_sqft']=size_rent['rent_per_month']/size_rent['size']
size_rent

,location,size,rent_per_month,price_per_sqft
1,Nairobi Cbd,86.5,53000.0,612.716763
2,Westlands,74.7,47000.0,629.183400
0,Kilimani,50.0,39000.0,780.000000


Kilimani has the highest price_per_sqft of all locations

2:Highest rent paying tenats

In [92]:
user_leases=pd.merge(lease_units_property_location,tenants,left_on='tenant_id',right_on='id').drop(['id'],axis=1)

In [93]:
user_rents=user_leases.groupby('name')['rent_paid'].sum().sort_values(ascending=False).reset_index()
user_rents

,name,rent_paid
0,Carol Wanjiru,2590000.0
1,Amina Mwangi,1305000.0
2,George Ouma,1254000.0
3,Brian Otieno,1155000.0
4,Hannah Achieng,1020000.0
5,David Kiptoo,810000.0
6,Joy Wambui,564000.0
7,Eunice Njeri,480000.0
8,Ian Wachira,372000.0
9,Farah Hassan,0.0


factoring in no of units they own

In [95]:
user_rents=user_leases.groupby('name').agg(rent_paid=('rent_paid','mean'),no_units=('tenant_id','count')).reset_index()
user_rents.sort_values(by='rent_paid',ascending=False,inplace=True,ignore_index=True)
user_rents

,name,rent_paid,no_units
0,Amina Mwangi,1305000.0,1
1,Carol Wanjiru,1295000.0,2
2,George Ouma,1254000.0,1
3,Hannah Achieng,1020000.0,1
4,David Kiptoo,810000.0,1
5,Brian Otieno,577500.0,2
6,Joy Wambui,564000.0,1
7,Eunice Njeri,480000.0,1
8,Ian Wachira,372000.0,1
9,Farah Hassan,0.0,1


3:Tenants with the highest Arrears

In [96]:
user_arrears=user_leases.groupby('name')['arrears'].sum().sort_values(ascending=False).reset_index()
user_arrears

,name,arrears
0,George Ouma,12000.0
1,Eunice Njeri,8000.0
2,Brian Otieno,6000.0
3,Joy Wambui,3000.0
4,Carol Wanjiru,2000.0
5,David Kiptoo,1000.0
6,Amina Mwangi,0.0
7,Farah Hassan,0.0
8,Hannah Achieng,0.0
9,Ian Wachira,0.0
